# Data Preprocessing
## Dataset: All_Structures_Combined.xlsx
### Outputs: Voc, Jsc, FF, PCE
### Split: 80% Train / 20% Test (random split)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Core scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# XGBoost and LightGBM
from xgboost import XGBRegressor, plot_importance
# import lightgbm as lgb

# Suppress warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## Load Dataset

In [2]:
df = pd.read_excel('All_Structures_Combined.xlsx')

print('Shape:', df.shape)
df.head()

Shape: (1800, 37)


,Structure,Window,W_Thickness,W_Eg,W_Ea,W_Dp,W_Nc,W_Nv,W_Ue,W_Uh,...,B_Nc,B_Nv,B_Ue,B_Uh,B_Na,B_dd,Voc,Jsc,FF,PCE
0,ZnS-CdTe-Sb2Se3,ZnS,0.05,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133306,27.169946,89.0159,27.4097
1,ZnS-CdTe-Sb2Se3,ZnS,0.10,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133291,27.154931,89.0156,27.3941
2,ZnS-CdTe-Sb2Se3,ZnS,0.15,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133273,27.137083,89.0154,27.3756
3,ZnS-CdTe-Sb2Se3,ZnS,0.20,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133254,27.118020,89.0153,27.3558
4,ZnS-CdTe-Sb2Se3,ZnS,0.25,3.5,4.2,10.0,1500000000000000000,1800000000000000000,50,20,...,2200000000000000000,18000000000000000000,15,5.1,10000000000000000000,100000000000000,1.133234,27.098420,89.0151,27.3355


## Encode Categorical Columns (Window, BSF)

In [3]:
# Show unique categorical values
print('Window unique values:', df['Window'].unique())
print('BSF unique values:', df['BSF'].unique())
print('Structure unique values:', df['Structure'].unique())

Window unique values: <StringArray>
['ZnS', 'CdS', 'In2S3']
Length: 3, dtype: str
BSF unique values: <StringArray>
['Sb2Se3', 'MoS2', 'ZnTe']
Length: 3, dtype: str
Structure unique values: <StringArray>
[  'ZnS-CdTe-Sb2Se3',     'ZnS-CdTe-MoS2',     'ZnS-CdTe-ZnTe',
   'CdS-CdTe-Sb2Se3',     'CdS-CdTe-MoS2',     'CdS-CdTe-ZnTe',
 'In2S3-CdTe-Sb2Se3',   'In2S3-CdTe-MoS2',   'In2S3-CdTe-ZnTe']
Length: 9, dtype: str


In [4]:
# Encode Window column
window_map = {
    'ZnS'  : 1,
    'CdS'  : 2,
    'In2S3': 3,
}
df['Window'] = df['Window'].map(window_map)

# Encode BSF column
bsf_map = {
    'Sb2Se3': 1,
    'MoS2'  : 2,
    'ZnTe'   : 3,
}
df['BSF'] = df['BSF'].map(bsf_map)

print('Window encoded:', df['Window'].unique())
print('BSF encoded   :', df['BSF'].unique())

Window encoded: [1 2 3]
BSF encoded   : [1 2 3]


## Drop Structure Column & Inspect

In [5]:
# Drop the Structure name column (not an input feature)
df.drop(columns=['Structure'], inplace=True)

print('Columns after dropping Structure:')
print(df.columns.tolist())

Columns after dropping Structure:
['Window', 'W_Thickness', 'W_Eg', 'W_Ea', 'W_Dp', 'W_Nc', 'W_Nv', 'W_Ue', 'W_Uh', 'W_Na', 'W_dd', 'A_Thickness', 'A_Eg', 'A_Ea', 'A_Dp', 'A_Nc', 'A_Nv', 'A_Ue', 'A_Uh', 'A_Na', 'A_dd', 'BSF', 'B_Thickness', 'B_Eg', 'B_Ea', 'B_Dp', 'B_Nc', 'B_Nv', 'B_Ue', 'B_Uh', 'B_Na', 'B_dd', 'Voc', 'Jsc', 'FF', 'PCE']


In [6]:
df.describe()

,Window,W_Thickness,W_Eg,W_Ea,W_Dp,W_Ue,W_Uh,W_dd,A_Thickness,A_Eg,...,B_Eg,B_Ea,B_Dp,B_Ue,B_Uh,B_dd,Voc,Jsc,FF,PCE
count,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1.800000e+03,1800.00,1800.00,...,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1.800000e+03,1800.000000,1800.000000,1800.000000,1800.000000
mean,2.000000,0.108750,2.757500,4.264583,11.983333,86.041667,24.916667,4.283300e+14,1.24,1.53,...,1.789417,3.854083,13.430500,34.341667,25.585000,4.283300e+14,1.098078,26.246351,86.368731,26.033494
std,0.816723,0.049873,0.534145,0.097793,5.058139,30.421551,9.706290,3.600632e+15,0.00,0.00,...,0.324208,0.142198,5.843275,29.073376,39.637942,3.600632e+15,0.169956,4.039388,13.206024,4.232512
min,1.000000,0.050000,2.000000,4.000000,5.000000,25.000000,10.000000,1.000000e+12,1.24,1.53,...,1.300000,3.400000,5.000000,5.000000,2.800000,1.000000e+12,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.100000,2.350000,4.200000,10.000000,50.000000,20.000000,1.000000e+14,1.24,1.53,...,1.530000,3.730000,10.000000,12.000000,2.800000,1.000000e+14,1.127736,26.604575,88.606600,26.647525
50%,2.000000,0.100000,2.400000,4.200000,10.000000,100.000000,25.000000,1.000000e+14,1.24,1.53,...,1.620000,3.800000,10.070000,15.000000,5.100000,1.000000e+14,1.132950,26.986382,88.639100,27.025300
75%,3.000000,0.100000,3.500000,4.400000,13.500000,100.000000,25.000000,1.000000e+14,1.24,1.53,...,2.260000,4.040000,18.000000,70.000000,50.000000,1.000000e+14,1.136463,27.073530,88.972400,27.274100
max,3.000000,0.500000,3.800000,4.450000,50.000000,250.000000,100.000000,5.000000e+16,1.24,1.53,...,2.260000,4.210000,50.000000,140.000000,280.000000,5.000000e+16,1.137285,30.548327,89.076600,28.394800


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 36 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Window       1800 non-null   int64  
 1   W_Thickness  1800 non-null   float64
 2   W_Eg         1800 non-null   float64
 3   W_Ea         1800 non-null   float64
 4   W_Dp         1800 non-null   float64
 5   W_Nc         1800 non-null   object 
 6   W_Nv         1800 non-null   object 
 7   W_Ue         1800 non-null   int64  
 8   W_Uh         1800 non-null   int64  
 9   W_Na         1800 non-null   object 
 10  W_dd         1800 non-null   int64  
 11  A_Thickness  1800 non-null   float64
 12  A_Eg         1800 non-null   float64
 13  A_Ea         1800 non-null   float64
 14  A_Dp         1800 non-null   float64
 15  A_Nc         1800 non-null   int64  
 16  A_Nv         1800 non-null   int64  
 17  A_Ue         1800 non-null   int64  
 18  A_Uh         1800 non-null   int64  
 19  A_Na         1800

## Convert All Numeric Columns to float64

In [8]:
# Convert all columns to float64 (Window and BSF are already int after mapping)
for col in df.columns:
    df[col] = df[col].astype('float64')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 36 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Window       1800 non-null   float64
 1   W_Thickness  1800 non-null   float64
 2   W_Eg         1800 non-null   float64
 3   W_Ea         1800 non-null   float64
 4   W_Dp         1800 non-null   float64
 5   W_Nc         1800 non-null   float64
 6   W_Nv         1800 non-null   float64
 7   W_Ue         1800 non-null   float64
 8   W_Uh         1800 non-null   float64
 9   W_Na         1800 non-null   float64
 10  W_dd         1800 non-null   float64
 11  A_Thickness  1800 non-null   float64
 12  A_Eg         1800 non-null   float64
 13  A_Ea         1800 non-null   float64
 14  A_Dp         1800 non-null   float64
 15  A_Nc         1800 non-null   float64
 16  A_Nv         1800 non-null   float64
 17  A_Ue         1800 non-null   float64
 18  A_Uh         1800 non-null   float64
 19  A_Na         1800

In [9]:
# Check max PCE value
print('Max PCE:', df['PCE'].max())
print('Max Voc:', df['Voc'].max())
print('Max Jsc:', df['Jsc'].max())
print('Max FF :', df['FF'].max())

Max PCE: 28.3948
Max Voc: 1.137285
Max Jsc: 30.5483268
Max FF : 89.0766


# **Train Test Split (80 / 20)**

In [10]:
# Define output (target) columns
target_cols = ['Voc', 'Jsc', 'FF', 'PCE']

# Separate features and targets
X_all = df.drop(columns=target_cols)
y_all = df[target_cols]

print('Feature matrix shape :', X_all.shape)
print('Target matrix shape  :', y_all.shape)
print('\nFeature columns:')
print(X_all.columns.tolist())

Feature matrix shape : (1800, 32)
Target matrix shape  : (1800, 4)

Feature columns:
['Window', 'W_Thickness', 'W_Eg', 'W_Ea', 'W_Dp', 'W_Nc', 'W_Nv', 'W_Ue', 'W_Uh', 'W_Na', 'W_dd', 'A_Thickness', 'A_Eg', 'A_Ea', 'A_Dp', 'A_Nc', 'A_Nv', 'A_Ue', 'A_Uh', 'A_Na', 'A_dd', 'BSF', 'B_Thickness', 'B_Eg', 'B_Ea', 'B_Dp', 'B_Nc', 'B_Nv', 'B_Ue', 'B_Uh', 'B_Na', 'B_dd']


In [30]:
# # ===== APPLY LOG10 TRANSFORMATION TO LARGE NUMBERS (>1e4) =====
# print('\n' + '='*80)
# print('🔧 APPLYING LOG10 TRANSFORMATION TO LARGE NUMBERS (>1e4)')
# print('='*80)

# # Identify columns with max values > 1e4 in numeric columns only
# numeric_cols = X_all.select_dtypes(include=[np.number]).columns
# log_transform_cols = [col for col in numeric_cols if X_all[col].max() > 1e4]

# print(f'\n❌ BEFORE TRANSFORMATION:')
# print(f'   Columns to transform (max > 1e4): {log_transform_cols}')
# for col in log_transform_cols:
#     print(f'   {col}: {X_all[col].min():.2e} to {X_all[col].max():.2e}')

# # Apply log10 transformation to identified columns
# for col in log_transform_cols:
#     X_all[col] = np.log10(X_all[col])

# print(f'\n✅ AFTER TRANSFORMATION:')
# for col in log_transform_cols:
#     print(f'   {col}: {X_all[col].min():.2f} to {X_all[col].max():.2f}')

# print(f'\n💡 BENEFITS:')
# print(f'   ✓ Large values normalized to manageable scale')
# print(f'   ✓ Better model performance with normalized features')
# print('='*80)

In [11]:
# 80/20 random split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=42
)

# Reset indices
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test  shape:', y_test.shape)

X_train shape: (1440, 32)
X_test  shape: (360, 32)
y_train shape: (1440, 4)
y_test  shape: (360, 4)


In [12]:
X_test.head()

,Window,W_Thickness,W_Eg,W_Ea,W_Dp,W_Nc,W_Nv,W_Ue,W_Uh,W_Na,...,B_Thickness,B_Eg,B_Ea,B_Dp,B_Nc,B_Nv,B_Ue,B_Uh,B_Na,B_dd
0,3.0,0.1,2.35,4.4,13.5,2.200000e+17,1.800000e+19,100.0,25.0,1.000000e+18,...,0.2,1.62,3.80,10.07,2.800000e+19,1.000000e+19,12.0,2.8,1.000000e+19,5.000000e+12
1,2.0,0.1,2.40,4.2,10.0,2.200000e+18,1.800000e+19,100.0,25.0,1.000000e+18,...,0.2,1.62,3.80,10.07,5.000000e+17,1.000000e+19,12.0,2.8,1.000000e+19,1.000000e+14
2,2.0,0.1,2.40,4.2,10.0,2.200000e+18,1.800000e+19,250.0,25.0,1.000000e+18,...,0.2,1.62,3.80,10.07,2.800000e+19,1.000000e+19,12.0,2.8,1.000000e+19,1.000000e+14
3,1.0,0.1,3.50,4.2,10.0,1.500000e+18,1.800000e+18,50.0,20.0,1.000000e+18,...,0.2,1.53,4.04,18.00,2.200000e+18,1.800000e+19,35.0,5.1,1.000000e+19,1.000000e+14
4,3.0,0.1,2.35,4.4,13.5,2.200000e+17,1.800000e+19,100.0,20.0,1.000000e+18,...,0.2,1.53,4.04,18.00,2.200000e+18,1.800000e+19,15.0,5.1,1.000000e+19,1.000000e+14


In [13]:
from sklearn.model_selection import KFold

# KFold cross-validation setup (5-fold)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# <font color="lightgreen"> **Random Forest Regressor (RF)**

In [14]:
from sklearn.ensemble import RandomForestRegressor

# Define hyperparameter grid for RF
rf_param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [10, 15, 20],
    # 'min_samples_split': [2, 5],
    # 'min_samples_leaf': [1, 2, 4]
}

# Initialize and tune model
rf = RandomForestRegressor(random_state=42)
rf_grid_search = GridSearchCV(rf, rf_param_grid, cv=kfold, scoring='neg_mean_squared_error', n_jobs=-1)
rf_grid_search.fit(X_train, y_train)

# Best RF model
best_rf = rf_grid_search.best_estimator_

# --- Test Evaluation ---
rf_y_pred = best_rf.predict(X_test)
rf_y_pred_df = pd.DataFrame(rf_y_pred, columns=target_cols)

rf_mse  = mean_squared_error(y_test, rf_y_pred_df, multioutput='raw_values')
rf_rmse = np.sqrt(rf_mse)
rf_mae  = mean_absolute_error(y_test, rf_y_pred_df, multioutput='raw_values')
rf_r2   = r2_score(y_test, rf_y_pred_df, multioutput='raw_values')

metrics_df_RF = pd.DataFrame({'MSE': rf_mse, 'RMSE': rf_rmse, 'MAE': rf_mae, 'R2 Score': rf_r2},
                              index=target_cols)

# --- Train Evaluation ---
rf_y_train_pred    = best_rf.predict(X_train)
rf_y_train_pred_df = pd.DataFrame(rf_y_train_pred, columns=target_cols)

rf_train_mse  = mean_squared_error(y_train, rf_y_train_pred_df, multioutput='raw_values')
rf_train_rmse = np.sqrt(rf_train_mse)
rf_train_mae  = mean_absolute_error(y_train, rf_y_train_pred_df, multioutput='raw_values')
rf_train_r2   = r2_score(y_train, rf_y_train_pred_df, multioutput='raw_values')

metrics_df_RF_train = pd.DataFrame({'MSE': rf_train_mse, 'RMSE': rf_train_rmse, 'MAE': rf_train_mae, 'R2 Score': rf_train_r2},
                                    index=target_cols)

print('Random Forest Test Metrics:\n', metrics_df_RF)
print('\nRandom Forest Train Metrics:\n', metrics_df_RF_train)

Random Forest Test Metrics:
            MSE      RMSE       MAE  R2 Score
Voc   0.002913  0.053975  0.009936  0.911298
Jsc   1.813520  1.346670  0.211270  0.904677
FF   21.416298  4.627775  0.810717  0.894349
PCE   2.192888  1.480840  0.380839  0.894785

Random Forest Train Metrics:
           MSE      RMSE       MAE  R2 Score
Voc  0.000267  0.016342  0.004744  0.990419
Jsc  0.178206  0.422145  0.091860  0.988596
FF   1.766173  1.328974  0.244524  0.989433
PCE  0.230755  0.480370  0.180418  0.986559


In [16]:
print('Best Parameters RF:', rf_grid_search.best_params_)

Best Parameters RF: {'max_depth': 15, 'n_estimators': 200}


# <font color="lightgreen"> **Gradient Boosting Regressor (GBR)**

In [17]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

# Define hyperparameter grid for GBR
gbr_param_grid = {
    'estimator__n_estimators'  : [200, 300, 500],
    'estimator__learning_rate' : [0.01, 0.05, 0.1],
    'estimator__max_depth'     : [2, 3, 4, 5, 6],
}

# Initialize and tune model using MultiOutputRegressor
gbr = MultiOutputRegressor(GradientBoostingRegressor(random_state=42))
gbr_grid_search = GridSearchCV(gbr, gbr_param_grid, cv=kfold, scoring='neg_mean_squared_error', n_jobs=-1)
gbr_grid_search.fit(X_train, y_train)

# Best GBR model
best_gbr = gbr_grid_search.best_estimator_

# --- Test Evaluation ---
gbr_y_pred    = best_gbr.predict(X_test)
gbr_y_pred_df = pd.DataFrame(gbr_y_pred, columns=target_cols)

gbr_mse  = mean_squared_error(y_test, gbr_y_pred_df, multioutput='raw_values')
gbr_rmse = np.sqrt(gbr_mse)
gbr_mae  = mean_absolute_error(y_test, gbr_y_pred_df, multioutput='raw_values')
gbr_r2   = r2_score(y_test, gbr_y_pred_df, multioutput='raw_values')

metrics_df_GBR = pd.DataFrame({'MSE': gbr_mse, 'RMSE': gbr_rmse, 'MAE': gbr_mae, 'R2 Score': gbr_r2},
                               index=target_cols)

# --- Train Evaluation ---
gbr_y_train_pred    = best_gbr.predict(X_train)
gbr_y_train_pred_df = pd.DataFrame(gbr_y_train_pred, columns=target_cols)

gbr_train_mse  = mean_squared_error(y_train, gbr_y_train_pred_df, multioutput='raw_values')
gbr_train_rmse = np.sqrt(gbr_train_mse)
gbr_train_mae  = mean_absolute_error(y_train, gbr_y_train_pred_df, multioutput='raw_values')
gbr_train_r2   = r2_score(y_train, gbr_y_train_pred_df, multioutput='raw_values')

metrics_df_GBR_train = pd.DataFrame({'MSE': gbr_train_mse, 'RMSE': gbr_train_rmse, 'MAE': gbr_train_mae, 'R2 Score': gbr_train_r2},
                                     index=target_cols)

print('Gradient Boosting Test Metrics:\n', metrics_df_GBR)
print('\nGradient Boosting Train Metrics:\n', metrics_df_GBR_train)

Gradient Boosting Test Metrics:
           MSE      RMSE       MAE  R2 Score
Voc  0.000015  0.003834  0.001469  0.999552
Jsc  0.052412  0.228936  0.054791  0.997245
FF   1.291591  1.136482  0.184486  0.993628
PCE  0.120947  0.347774  0.114214  0.994197

Gradient Boosting Train Metrics:
           MSE      RMSE       MAE  R2 Score
Voc  0.000004  0.001882  0.000895  0.999873
Jsc  0.006959  0.083420  0.030517  0.999555
FF   0.047406  0.217730  0.078512  0.999716
PCE  0.032835  0.181204  0.076531  0.998087


In [19]:
print('Best Parameters GB:', gbr_grid_search.best_params_)

Best Parameters GB: {'estimator__learning_rate': 0.1, 'estimator__max_depth': 3, 'estimator__n_estimators': 500}


# <font color="lightgreen"> **Linear Regression**

In [20]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(X_train)
x_test = scaler.transform(X_test)  # Use transform, not fit_transform

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

In [50]:
# Train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predictions
lr_y_pred = lr.predict(X_test)

# Convert predictions to DataFrame
lr_y_pred_df = pd.DataFrame(lr_y_pred, columns=['Voc', 'Jsc', 'FF', 'PCE'])

# Evaluation on scaled targets
lr_mse = mean_squared_error(y_test, lr_y_pred_df, multioutput='raw_values')
lr_rmse = np.sqrt(lr_mse)
lr_mae = mean_absolute_error(y_test, lr_y_pred_df, multioutput='raw_values')
lr_r2 = r2_score(y_test, lr_y_pred_df, multioutput='raw_values')

# Print metrics
metrics_df_lr = pd.DataFrame({'MSE': lr_mse, 'RMSE': lr_rmse, 'MAE': lr_mae, 'R2 Score': lr_r2},
                           index=['Voc', 'Jsc', 'FF', 'PCE'])
print("Linear Regression Test Metrics (Scaled):\n", metrics_df_lr)

# Also show train metrics
lr_y_train_pred = lr.predict(X_train)
lr_y_train_pred_df = pd.DataFrame(lr_y_train_pred, columns=['Voc', 'Jsc', 'FF', 'PCE'])

lr_train_mse = mean_squared_error(y_train, lr_y_train_pred_df, multioutput='raw_values')
lr_train_rmse = np.sqrt(lr_train_mse)
lr_train_mae = mean_absolute_error(y_train, lr_y_train_pred_df, multioutput='raw_values')
lr_train_r2 = r2_score(y_train, lr_y_train_pred_df, multioutput='raw_values')

metrics_df_lr_train = pd.DataFrame({'MSE': lr_train_mse, 'RMSE': lr_train_rmse, 'MAE': lr_train_mae, 'R2 Score': lr_train_r2}, index=['Voc', 'Jsc', 'FF', 'PCE'])

print("\nLinear Regression Train Metrics (Scaled):\n", metrics_df_lr_train)

Linear Regression Test Metrics (Scaled):
             MSE       RMSE       MAE  R2 Score
Voc    0.032747   0.180962  0.065220  0.002935
Jsc   18.977464   4.356313  1.410982  0.002502
FF   202.131396  14.217292  4.692539  0.002847
PCE   20.760310   4.556348  1.818568  0.003915

Linear Regression Train Metrics (Scaled):
             MSE       RMSE       MAE  R2 Score
Voc    0.027784   0.166687  0.060924  0.003178
Jsc   15.599164   3.949578  1.278139  0.001738
FF   166.658440  12.909626  4.199988  0.002910
PCE   17.122684   4.137957  1.688557  0.002646


# <font color="lightgreen"> **Decision Tree Regressor (DT)**

In [24]:
# Defining hyperparameter grid
dt_param_grid = {
    'max_depth'        : [3, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4]
}

# Initializing and tuning model
dt = DecisionTreeRegressor(random_state=42)
dt_grid_search = GridSearchCV(dt, dt_param_grid, cv=kfold, scoring='neg_mean_squared_error', n_jobs=-1)
dt_grid_search.fit(X_train, y_train)

# Best model
best_dt = dt_grid_search.best_estimator_

# --- Test Evaluation ---
dt_y_pred    = best_dt.predict(X_test)
dt_y_pred_df = pd.DataFrame(dt_y_pred, columns=target_cols)

dt_mse  = mean_squared_error(y_test, dt_y_pred_df, multioutput='raw_values')
dt_rmse = np.sqrt(dt_mse)
dt_mae  = mean_absolute_error(y_test, dt_y_pred_df, multioutput='raw_values')
dt_r2   = r2_score(y_test, dt_y_pred_df, multioutput='raw_values')

metrics_df_DT = pd.DataFrame({'MSE': dt_mse, 'RMSE': dt_rmse, 'MAE': dt_mae, 'R2 Score': dt_r2},
                              index=target_cols)

# --- Train Evaluation ---
dt_y_train_pred    = best_dt.predict(X_train)
dt_y_train_pred_df = pd.DataFrame(dt_y_train_pred, columns=target_cols)

dt_train_mse  = mean_squared_error(y_train, dt_y_train_pred_df, multioutput='raw_values')
dt_train_rmse = np.sqrt(dt_train_mse)
dt_train_mae  = mean_absolute_error(y_train, dt_y_train_pred_df, multioutput='raw_values')
dt_train_r2   = r2_score(y_train, dt_y_train_pred_df, multioutput='raw_values')

metrics_df_DT_train = pd.DataFrame({'MSE': dt_train_mse, 'RMSE': dt_train_rmse, 'MAE': dt_train_mae, 'R2 Score': dt_train_r2},
                                    index=target_cols)

print('Decision Tree Test Metrics:\n', metrics_df_DT)
print('\nDecision Tree Train Metrics:\n', metrics_df_DT_train)

Decision Tree Test Metrics:
            MSE      RMSE       MAE  R2 Score
Voc   0.007830  0.088487  0.015314  0.761597
Jsc   4.669679  2.160944  0.303225  0.754551
FF   49.630962  7.044925  1.122808  0.755161
PCE   5.346985  2.312355  0.581761  0.743450

Decision Tree Train Metrics:
           MSE      RMSE       MAE  R2 Score
Voc  0.001091  0.033028  0.008581  0.960863
Jsc  0.826591  0.909170  0.168825  0.947103
FF   6.284396  2.506870  0.471959  0.962402
PCE  1.042733  1.021143  0.359822  0.939263


In [23]:
print('Best Parameters DT:', dt_grid_search.best_params_)

Best Parameters DT: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}


# <font color="lightgreen"> **XGBoost Regressor**

## Basic XGBRegressor

In [37]:
# Initializing XGBoost Regressor
xgb_reg = XGBRegressor(objective='reg:squarederror')

# Training the model
xgb_reg.fit(X_train, y_train)

# Predictions
Xgb_y_pred    = xgb_reg.predict(X_test)
Xgb_y_pred_df = pd.DataFrame(Xgb_y_pred, columns=target_cols)

# Evaluation
mse  = mean_squared_error(y_test, Xgb_y_pred_df, multioutput='raw_values')
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, Xgb_y_pred_df, multioutput='raw_values')
r2   = r2_score(y_test, Xgb_y_pred_df, multioutput='raw_values')

metrics_df_xgb = pd.DataFrame({'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2},
                               index=target_cols)
print(metrics_df_xgb)

          MSE      RMSE       MAE  R2 Score
Voc  0.000019  0.004336  0.001310  0.999428
Jsc  0.034324  0.185268  0.034251  0.998196
FF   0.127879  0.357602  0.083491  0.999369
PCE  0.022316  0.149384  0.056136  0.998929


## With Hyperparameter Tuning

In [46]:
# Hyperparameter tuning using GridSearchCV
xgb_param_grid = {
    'n_estimators'    : [100, 200, 300, 500, 1000],
    'learning_rate'   : [ 0.05, 0.1, 0.2],
    'max_depth'       : [2, 3, 4, 5, 6],
    # 'subsample'       : [0.8, 1.0],
    # 'colsample_bytree': [0.8, 1.0],
}

grid_search_xgb = GridSearchCV(XGBRegressor(objective='reg:squarederror'), xgb_param_grid,
                                cv=kfold, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
grid_search_xgb.fit(X_train, y_train)

# Best model from GridSearchCV
best_model_XGB = grid_search_xgb.best_estimator_
print('Best Parameters:', grid_search_xgb.best_params_)

# --- Test Evaluation ---
y_pred_XGB    = best_model_XGB.predict(X_test)
y_pred_XGB_df = pd.DataFrame(y_pred_XGB, columns=target_cols)

mse  = mean_squared_error(y_test, y_pred_XGB_df, multioutput='raw_values')
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_XGB_df, multioutput='raw_values')
r2   = r2_score(y_test, y_pred_XGB_df, multioutput='raw_values')

metrics_df_XGB = pd.DataFrame({'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2},
                               index=target_cols)

# --- Train Evaluation ---
y_train_pred    = best_model_XGB.predict(X_train)
y_train_pred_df = pd.DataFrame(y_train_pred, columns=target_cols)

mse_train  = mean_squared_error(y_train, y_train_pred_df, multioutput='raw_values')
rmse_train = np.sqrt(mse_train)
mae_train  = mean_absolute_error(y_train, y_train_pred_df, multioutput='raw_values')
r2_train   = r2_score(y_train, y_train_pred_df, multioutput='raw_values')

metrics_train_df_XGB = pd.DataFrame({'MSE': mse_train, 'RMSE': rmse_train, 'MAE': mae_train, 'R2 Score': r2_train},
                                     index=target_cols)

print('XGBoost Test Metrics:\n', metrics_df_XGB)
print('\nXGBoost Train Metrics:\n', metrics_train_df_XGB)

Fitting 5 folds for each of 75 candidates, totalling 375 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000}
XGBoost Test Metrics:
           MSE      RMSE       MAE  R2 Score
Voc  0.000016  0.004025  0.001535  0.999507
Jsc  0.032918  0.181432  0.039450  0.998270
FF   0.991508  0.995745  0.118501  0.995109
PCE  0.074837  0.273563  0.093541  0.996409

XGBoost Train Metrics:
           MSE      RMSE       MAE  R2 Score
Voc  0.000004  0.002109  0.001018  0.999840
Jsc  0.002762  0.052554  0.019461  0.999823
FF   0.008095  0.089973  0.035728  0.999952
PCE  0.012377  0.111251  0.052705  0.999279


In [47]:
print('Best Parameters XGB:', grid_search_xgb.best_params_)

Best Parameters XGB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 1000}


# <font color="lightblue"> **LightGBM - Efficient Gradient Boosting**

In [35]:
# # ========================================================================
# # LIGHTGBM - HYPERPARAMETER TUNING WITH MULTIOUTPUT
# # ========================================================================

# print('='*80)
# print('🔍 LIGHTGBM HYPERPARAMETER TUNING - UNIFIED MODEL')
# print('='*80)

# import lightgbm as lgb

# # Define hyperparameter grid for LightGBM
# lgb_param_grid = {
#     'estimator__n_estimators' : [100, 300, 500],
#     'estimator__learning_rate': [0.01, 0.05, 0.1],
#     'estimator__max_depth'    : [4, 5, 6],
#     # 'estimator__num_leaves'  : [31, 63, 127],
#     # 'estimator__subsample'   : [0.8, 1.0],
#     # 'estimator__colsample_bytree': [0.8, 1.0]
# }

# # Initialize and tune model using MultiOutputRegressor
# lgb_model = MultiOutputRegressor(lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1))
# lgb_grid_search = GridSearchCV(lgb_model, lgb_param_grid, cv=kfold, scoring='neg_mean_squared_error', n_jobs=-1, verbose=0)
# lgb_grid_search.fit(X_train, y_train)

# # Best LGB model
# best_lgb = lgb_grid_search.best_estimator_

# print('\n✅ Best Parameters for LightGBM:')
# for param, value in lgb_grid_search.best_params_.items():
#     print(f'   • {param}: {value}')
# print(f'   • Best CV Score: {-lgb_grid_search.best_score_:.6f} (MSE)')

# # --- Test Evaluation ---
# lgb_y_pred    = best_lgb.predict(X_test)
# lgb_y_pred_df = pd.DataFrame(lgb_y_pred, columns=target_cols)

# lgb_mse  = mean_squared_error(y_test, lgb_y_pred_df, multioutput='raw_values')
# lgb_rmse = np.sqrt(lgb_mse)
# lgb_mae  = mean_absolute_error(y_test, lgb_y_pred_df, multioutput='raw_values')
# lgb_r2   = r2_score(y_test, lgb_y_pred_df, multioutput='raw_values')

# metrics_df_LGB = pd.DataFrame({'MSE': lgb_mse, 'RMSE': lgb_rmse, 'MAE': lgb_mae, 'R2 Score': lgb_r2},
#                                index=target_cols)

# # --- Train Evaluation ---
# lgb_y_train_pred    = best_lgb.predict(X_train)
# lgb_y_train_pred_df = pd.DataFrame(lgb_y_train_pred, columns=target_cols)

# lgb_train_mse  = mean_squared_error(y_train, lgb_y_train_pred_df, multioutput='raw_values')
# lgb_train_rmse = np.sqrt(lgb_train_mse)
# lgb_train_mae  = mean_absolute_error(y_train, lgb_y_train_pred_df, multioutput='raw_values')
# lgb_train_r2   = r2_score(y_train, lgb_y_train_pred_df, multioutput='raw_values')

# metrics_df_LGB_train = pd.DataFrame({'MSE': lgb_train_mse, 'RMSE': lgb_train_rmse, 'MAE': lgb_train_mae, 'R2 Score': lgb_train_r2},
#                                      index=target_cols)

# print('\n' + '='*80)
# print('📈 TEST SET PERFORMANCE - LIGHTGBM')
# print('='*80)
# print(metrics_df_LGB.to_string())

# print('\n' + '='*80)
# print('📈 TRAIN SET PERFORMANCE - LIGHTGBM')
# print('='*80)
# print(metrics_df_LGB_train.to_string())

# print('\n✅ Unified LightGBM model created and evaluated')

# <font color="lightgreen"> **Comparing Models according to PCE**

In [48]:
# Create a dictionary to store the PCE metrics for each model
pce_metrics = {
    'Linear Regression': metrics_df_lr.loc['PCE'],
    'Decision Tree'   : metrics_df_DT.loc['PCE'],
    'Gradient Boosting': metrics_df_GBR.loc['PCE'],
    'Random Forest'   : metrics_df_RF.loc['PCE'],
    'XGBoost'         : metrics_df_XGB.loc['PCE'],
    # 'LightGBM'        : metrics_df_LGB.loc['PCE'],
}

# Create a DataFrame from the dictionary
comparison_df = pd.DataFrame(pce_metrics).T
comparison_df

,MSE,RMSE,MAE,R2 Score
Linear Regression,20.760310,4.556348,1.818568,0.003915
Decision Tree,5.346985,2.312355,0.581761,0.743450
Gradient Boosting,0.120947,0.347774,0.114214,0.994197
Random Forest,2.192888,1.480840,0.380839,0.894785
XGBoost,0.074837,0.273563,0.093541,0.996409


In [49]:
# Full comparison across all targets
print('\n' + '='*80)
print('📊 R² SCORE SUMMARY — ALL TARGETS')
print('='*80)

r2_summary = pd.DataFrame({
    'Decision Tree'   : dt_r2,
    'Linear Regression': lr_r2,
    'Gradient Boosting': gbr_r2,
    'Random Forest'   : rf_r2,
    'XGBoost'         : r2,
    # 'LightGBM'        : lgb_r2,
}, index=target_cols)

print(r2_summary.to_string())


📊 R² SCORE SUMMARY — ALL TARGETS
     Decision Tree  Linear Regression  Gradient Boosting  Random Forest   XGBoost
Voc       0.761597           0.002935           0.999552       0.911298  0.999507
Jsc       0.754551           0.002502           0.997245       0.904677  0.998270
FF        0.755161           0.002847           0.993628       0.894349  0.995109
PCE       0.743450           0.003915           0.994197       0.894785  0.996409
